# NER Notebook 1: Inference & Rule-Based NER

**BSAN 6200 — Text Mining & Social Media Analytics**

## What You'll Learn
1. **Pretrained NER inference** using spaCy, NLTK, and HuggingFace Transformers
2. **Rule-based NER** using regex, dictionaries, and spaCy EntityRuler
3. **NER data formats**: CoNLL (IOB2) and JSONL
4. **Label Studio integration**: export annotations and import pre-labeled data

## Prerequisites
- Basic Python
- Familiarity with NLP concepts (tokenization, POS tagging)

---

---
## Part 1: Pretrained NER with spaCy

spaCy is an industrial-strength NLP library. Its pretrained models include NER
out of the box. We'll use `en_core_web_sm` (small English model).

**Key concepts:**
- spaCy processes text into a `Doc` object containing tokens, POS tags, and entities
- Entities have `.text` (the string), `.label_` (entity type), `.start_char` / `.end_char` (positions)
- displaCy is spaCy's built-in visualization tool

In [42]:
# ============================================================
# SETUP: Install and import spaCy
# ============================================================
# spaCy is pre-installed in Colab. If running locally:
# !pip install spacy
# !python -m spacy download en_core_web_sm

import spacy
from spacy import displacy

# Load the small English model
# This model was trained on web text (blogs, news, comments)
# It recognizes entities like PERSON, ORG, GPE, DATE, MONEY, etc.
nlp = spacy.load("en_core_web_sm")

print(f"Model loaded: {nlp.meta['name']}")
print(f"Pipeline components: {nlp.pipe_names}")

Model loaded: core_web_sm
Pipeline components: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


In [43]:
# ============================================================
# BASIC NER: Process a single sentence
# ============================================================
# nlp() processes text and returns a Doc object with all annotations

text = "Thomas Jefferson writes the Declaration of Independence"
doc = nlp(text)

# Loop through detected entities
# Each entity has:
#   .text       → the actual text span (e.g., "Thomas Jefferson")
#   .start_char → character position where entity starts
#   .end_char   → character position where entity ends
#   .label_     → entity type label (e.g., "PERSON")

print(f"Text: {text}")
print(f"{'Entity':<30} {'Start':>5} {'End':>5}  {'Label'}")
print("-" * 60)
for ent in doc.ents:
    print(f"{ent.text:<30} {ent.start_char:>5} {ent.end_char:>5}  {ent.label_}")

Text: Thomas Jefferson writes the Declaration of Independence
Entity                         Start   End  Label
------------------------------------------------------------
Thomas Jefferson                   0    16  PERSON
the Declaration of Independence    24    55  WORK_OF_ART


In [44]:
# ============================================================
# VISUALIZATION: Use displaCy to render entities inline
# ============================================================
# displaCy highlights entities directly in the text
# style="ent" → entity visualization (vs "dep" for dependency parsing)
# jupyter=True → render inline in notebook

displacy.render(doc, style="ent", jupyter=True, options={'distance': 120})

In [45]:
# ============================================================
# BATCH PROCESSING: Run NER on multiple sentences
# ============================================================
# Let's test with various sentences to see what spaCy catches
# and what it misses — this builds intuition for NER limitations

texts = [
    "Thomas Jefferson writes the Declaration of Independence",
    "My friend Tom spends his time google at work",        # lowercase 'google' = verb
    "My friend Tom works at Google",                        # capitalized 'Google' = ORG
    "Lin Manuel-Miranda writes Hamilton the musical",
    "Washington is a rainy state.",                          # GPE or PERSON?
    "Apple announced record revenue of $81.8 billion in Q3 2024",
    "Dr. Sarah Chen published a paper on deep learning at MIT",
]

for text in texts:
    doc = nlp(text)
    print(f"\nText: {text}")

    # Render the entity visualization
    displacy.render(doc, style="ent", jupyter=True, options={'distance': 120})

    # Also print entities in structured format
    for ent in doc.ents:
        print(f"  → {ent.text:<25} {ent.label_:<10} ({ent.start_char}-{ent.end_char})")

    if not doc.ents:
        print("  → No entities detected!")


Text: Thomas Jefferson writes the Declaration of Independence


  → Thomas Jefferson          PERSON     (0-16)
  → the Declaration of Independence WORK_OF_ART (24-55)

Text: My friend Tom spends his time google at work


  → Tom                       PERSON     (10-13)

Text: My friend Tom works at Google


  → Tom                       PERSON     (10-13)
  → Google                    ORG        (23-29)

Text: Lin Manuel-Miranda writes Hamilton the musical


  → Lin Manuel-Miranda        PERSON     (0-18)
  → Hamilton                  PERSON     (26-34)

Text: Washington is a rainy state.


  → Washington                GPE        (0-10)

Text: Apple announced record revenue of $81.8 billion in Q3 2024


  → Apple                     ORG        (0-5)
  → $81.8 billion             MONEY      (34-47)
  → Q3 2024                   DATE       (51-58)

Text: Dr. Sarah Chen published a paper on deep learning at MIT


  → Sarah Chen                PERSON     (4-14)
  → MIT                       ORG        (53-56)


### spaCy Entity Types Reference

| Label | Description | Example |
|-------|-------------|---------|
| PERSON | People (real or fictional) | "Thomas Jefferson" |
| NORP | Nationalities, religious/political groups | "American", "Republican" |
| ORG | Organizations | "Google", "MIT" |
| GPE | Countries, cities, states | "California", "Japan" |
| LOC | Non-GPE locations | "Mount Everest", "Pacific Ocean" |
| DATE | Dates or periods | "Q3 2024", "last Tuesday" |
| MONEY | Monetary values | "$81.8 billion" |
| PRODUCT | Objects, vehicles, foods | "iPhone", "Boeing 747" |
| EVENT | Named events | "World War II", "Olympics" |
| WORK_OF_ART | Titles of creative works | "Declaration of Independence" |

**Discussion**: Which entities did spaCy get right? Which did it miss? Why?

---
## Part 2: NER with NLTK

NLTK takes a different approach than spaCy:
1. First **tokenize** the text into words
2. Then **POS-tag** each token (noun, verb, etc.)
3. Then run **named entity chunking** on the POS-tagged tokens

This multi-step pipeline is more transparent but less accurate than modern models.

In [46]:
# ============================================================
# SETUP: Install NLTK and download required data
# ============================================================
import nltk

# Download the models NLTK needs for NER
# punkt_tab     → tokenizer model (splits text into words)
# averaged_perceptron_tagger_eng → POS tagger (labels words as noun, verb, etc.)
# maxent_ne_chunker_tab → NER model (identifies named entities)
# words         → word list (used by the NER model)

nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('maxent_ne_chunker_tab', quiet=True)
nltk.download('words', quiet=True)

print("NLTK data downloaded!")

NLTK data downloaded!


In [47]:
# ============================================================
# NLTK NER PIPELINE: Tokenize → POS Tag → NE Chunk
# ============================================================
# Unlike spaCy (which does everything in one call), NLTK requires
# you to run each step explicitly

text = "Thomas Jefferson writes the Declaration of Independence"

# Step 1: Tokenize — split text into individual words
tokens = nltk.word_tokenize(text)
print(f"Step 1 - Tokens: {tokens}")

# Step 2: POS Tag — label each word with its part of speech
# NNP = proper noun (singular), VBZ = verb (3rd person singular), DT = determiner
pos_tags = nltk.pos_tag(tokens)
print(f"Step 2 - POS Tags: {pos_tags}")

# Step 3: Named Entity Chunking — group POS-tagged words into entities
# Returns a tree structure where named entities are subtrees
ne_tree = nltk.ne_chunk(pos_tags)
print(f"Step 3 - NE Tree:")
print(ne_tree)

Step 1 - Tokens: ['Thomas', 'Jefferson', 'writes', 'the', 'Declaration', 'of', 'Independence']
Step 2 - POS Tags: [('Thomas', 'NNP'), ('Jefferson', 'NNP'), ('writes', 'VBZ'), ('the', 'DT'), ('Declaration', 'NNP'), ('of', 'IN'), ('Independence', 'NNP')]
Step 3 - NE Tree:
(S
  (PERSON Thomas/NNP)
  (PERSON Jefferson/NNP)
  writes/VBZ
  the/DT
  Declaration/NNP
  of/IN
  (GPE Independence/NNP))


In [48]:
# ============================================================
# EXTRACT ENTITIES FROM NLTK'S TREE STRUCTURE
# ============================================================
# NLTK returns a Tree object. Named entities are subtrees with
# a label (e.g., "PERSON", "ORGANIZATION").
# Non-entity words are just (word, POS) tuples at the top level.

texts = [
    "Thomas Jefferson writes the Declaration of Independence",
    "My friend Tom works at Google",
    "Lin Manuel-Miranda writes Hamilton the musical",
    "Washington is a rainy state.",
    "Apple announced record revenue of $81.8 billion in Q3 2024",
]

for text in texts:
    print(f"\nText: {text}")

    # Run the NLTK NER pipeline
    tokens = nltk.word_tokenize(text)
    pos_tags = nltk.pos_tag(tokens)
    ne_tree = nltk.ne_chunk(pos_tags)

    # Extract entities from the tree
    # Each subtree with a label is a named entity
    # The subtree's label() gives the entity type
    # The subtree's leaves() give the (word, POS) pairs
    entities = []
    for subtree in ne_tree:
        # Check if this node is a named entity (subtree) vs regular word (tuple)
        if hasattr(subtree, 'label'):
            entity_text = " ".join(word for word, pos in subtree.leaves())
            entity_type = subtree.label()
            entities.append((entity_text, entity_type))
            print(f"  → {entity_text:<25} {entity_type}")

    if not entities:
        print("  → No entities detected!")


Text: Thomas Jefferson writes the Declaration of Independence
  → Thomas                    PERSON
  → Jefferson                 PERSON
  → Independence              GPE

Text: My friend Tom works at Google
  → Tom                       PERSON
  → Google                    ORGANIZATION

Text: Lin Manuel-Miranda writes Hamilton the musical
  → Lin                       PERSON
  → Hamilton                  PERSON

Text: Washington is a rainy state.
  → Washington                GPE

Text: Apple announced record revenue of $81.8 billion in Q3 2024
  → Apple                     PERSON


### spaCy vs NLTK: Key Differences

| Aspect | spaCy | NLTK |
|--------|-------|------|
| Approach | End-to-end neural pipeline | Multi-step (tokenize → POS → chunk) |
| Speed | Fast (optimized C code) | Slower |
| Accuracy | Higher (modern models) | Lower (older algorithms) |
| Entity types | 18 types | ~3 types (PERSON, ORGANIZATION, GPE) |
| Best for | Production use | Teaching/understanding the pipeline |

**Takeaway**: Use spaCy for real projects, but NLTK helps you understand what's happening under the hood.

---
## Part 3: NER with HuggingFace Transformers

HuggingFace provides access to thousands of pretrained NER models.
These are deep learning models (BERT, DistilBERT, etc.) that are
significantly more accurate than spaCy or NLTK, especially for
domain-specific entities.

**Key concept**: The `pipeline("ner")` function wraps model loading,
tokenization, inference, and post-processing into a single call.

In [49]:
# ============================================================
# SETUP: Load a pretrained Transformer NER model
# ============================================================
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

# We use DistilBERT fine-tuned on CoNLL-2003 NER dataset
# This model recognizes: PER (person), ORG (organization),
#                         LOC (location), MISC (miscellaneous)
model_name = "dslim/distilbert-NER"

# The pipeline handles everything:
# 1. Tokenizes input text into subwords
# 2. Runs the model to get per-token predictions
# 3. Aggregates subword predictions back to word-level entities
# aggregation_strategy="simple" merges subword tokens like ["Wash", "##ington"]
ner_pipeline = pipeline(
    "ner",
    model=model_name,
    aggregation_strategy="simple"
)

print(f"Model loaded: {model_name}")
print("Entity types: PER (person), ORG (organization), LOC (location), MISC (miscellaneous)")

Device set to use cpu


Model loaded: dslim/distilbert-NER
Entity types: PER (person), ORG (organization), LOC (location), MISC (miscellaneous)


In [50]:
# ============================================================
# RUN TRANSFORMER NER ON OUR TEST SENTENCES
# ============================================================
# The pipeline returns a list of dicts, each with:
#   'word'         → the detected entity text
#   'entity_group' → the entity type (PER, ORG, LOC, MISC)
#   'score'        → confidence score (0-1)
#   'start'        → character start position
#   'end'          → character end position

texts = [
    "Thomas Jefferson writes the Declaration of Independence",
    "My friend Tom works at Google",
    "Apple announced record revenue of $81.8 billion in Q3 2024",
    "Dr. Sarah Chen published a paper on deep learning at MIT",
    "Lin Manuel-Miranda writes Hamilton the musical",
]

for text in texts:
    print(f"\nText: {text}")
    results = ner_pipeline(text)

    if results:
        for r in results:
            # Display each entity with its type and confidence score
            # Higher scores (closer to 1.0) mean more confident predictions
            print(f"  → {r['word']:<25} {r['entity_group']:<8} "
                  f"(confidence: {r['score']:.3f}, chars {r['start']}-{r['end']})")
    else:
        print("  → No entities detected!")


Text: Thomas Jefferson writes the Declaration of Independence


  → Thomas Jefferson          PER      (confidence: 0.997, chars 0-16)
  → Declaration of Independence MISC     (confidence: 0.962, chars 28-55)

Text: My friend Tom works at Google
  → Tom                       PER      (confidence: 0.989, chars 10-13)
  → Google                    ORG      (confidence: 0.946, chars 23-29)

Text: Apple announced record revenue of $81.8 billion in Q3 2024
  → Apple                     ORG      (confidence: 0.997, chars 0-5)

Text: Dr. Sarah Chen published a paper on deep learning at MIT
  → Sarah Chen                PER      (confidence: 0.995, chars 4-14)
  → MIT                       ORG      (confidence: 0.724, chars 53-56)

Text: Lin Manuel-Miranda writes Hamilton the musical
  → Lin Manuel - Miranda      PER      (confidence: 0.857, chars 0-18)
  → Hamilton                  PER      (confidence: 0.942, chars 26-34)


In [51]:
# ============================================================
# VISUALIZE TRANSFORMER RESULTS WITH DISPLACY
# ============================================================
# We can convert HuggingFace output to spaCy's displaCy format
# for nice inline visualization

for text in texts[:3]:
    results = ner_pipeline(text)

    # Convert to displaCy format:
    # displaCy expects a dict with "text", "ents" (list of entity spans), "title"
    ents = [{"start": r["start"], "end": r["end"], "label": r["entity_group"]}
            for r in results]

    doc_data = {"text": text, "ents": ents, "title": None}

    # manual=True tells displaCy we're providing pre-computed entities
    displacy.render(doc_data, style="ent", manual=True, jupyter=True)

---
## Part 4: Rule-Based NER with Regular Expressions

Regular expressions (regex) match text patterns. They're perfect for
**structured entities** like phone numbers, emails, dates, and product codes.

**When to use regex for NER:**
- Entities follow a predictable pattern
- You need 100% precision on known patterns
- The domain has well-defined formats (financial, medical codes, etc.)

### Quick Regex Reference

| Pattern | Meaning | Example |
|---------|---------|---------|
| `\d` | One digit | `\d{3}` → "202" |
| `[a-z]` | One lowercase letter | `[A-Z]{2}` → "CA" |
| `+` | One or more | `\w+` → "word" |
| `*` | Zero or more | `\d*` → "" or "123" |
| `?` | Zero or one | `-?` → optional dash |
| `{n}` | Exactly n times | `\d{4}` → "2024" |
| `(...)` | Capture group | `(\d+)` → extract number |
| `a\|b` | a or b | `USD\|\$` → either |

In [52]:
# ============================================================
# REGEX BASICS: The three core functions
# ============================================================
import re

# Define a regex pattern for US phone numbers: 3 digits - 3 digits - 4 digits
regex = re.compile(r"\d{3}-\d{3}-\d{4}")

# Test texts with various phone number formats
texts = [
    'If you want to call the White house, please use 202-456-1111',
    'If you want to call the White house, please use (202)456-1111',
    'If you want to call the White house, please use (202) 456-1111',
    'If you want to call the White house, please use 2024561111',
    'If you want to call the White house, please use 202.456.1111',
    'If you want to call the White house, please use 202 456 1111',
]

# ---- re.findall() ----
# Returns a LIST of all matches found in the text
print("=== re.findall() — find all matches ===")
for text in texts:
    matches = re.findall(regex, text)
    status = "✓ MATCH" if matches else "✗ no match"
    print(f"  {status}: {matches}  ← {text[-30:]}")

=== re.findall() — find all matches ===
  ✓ MATCH: ['202-456-1111']  ← house, please use 202-456-1111
  ✗ no match: []  ← ouse, please use (202)456-1111
  ✗ no match: []  ← use, please use (202) 456-1111
  ✗ no match: []  ← e house, please use 2024561111
  ✗ no match: []  ← house, please use 202.456.1111
  ✗ no match: []  ← house, please use 202 456 1111


In [53]:
# ---- re.split() ----
# Splits the text at each match, returning the pieces between matches
print("\n=== re.split() — split text at matches ===")
result = re.split(regex, texts[0])
print(f"  Original: {texts[0]}")
print(f"  Split:    {result}")

# ---- re.sub() ----
# Replaces matches with a substitute string
# This is KEY for de-identification: redacting PII from documents
print("\n=== re.sub() — replace matches (de-identification) ===")
result = re.sub(regex, "[REDACTED]", texts[0])
print(f"  Original:  {texts[0]}")
print(f"  Redacted:  {result}")

# You can limit replacements with count parameter
result = re.sub(regex, "[REDACTED]", texts[0], count=1)
print(f"  First only: {result}")


=== re.split() — split text at matches ===
  Original: If you want to call the White house, please use 202-456-1111
  Split:    ['If you want to call the White house, please use ', '']

=== re.sub() — replace matches (de-identification) ===
  Original:  If you want to call the White house, please use 202-456-1111
  Redacted:  If you want to call the White house, please use [REDACTED]
  First only: If you want to call the White house, please use [REDACTED]


In [54]:
# ============================================================
# EXERCISE: Build a regex that handles MULTIPLE phone formats
# ============================================================
# Real-world phone numbers come in many formats.
# Try matching: 202-456-1111, (202)456-1111, (202) 456-1111,
#               2024561111, 202.456.1111, +1-202-456-1111

# Step-by-step breakdown of the pattern:
# (?:\+1[\s-]?)?     → optional "+1" country code with optional space/dash
# \(?                 → optional opening parenthesis
# \d{3}              → 3-digit area code
# \)?                 → optional closing parenthesis
# [\s.-]?            → optional separator (space, dot, or dash)
# \d{3}              → 3-digit prefix
# [\s.-]?            → optional separator
# \d{4}              → 4-digit line number

flexible_regex = re.compile(r"(?:\+1[\s-]?)?\(?\d{3}\)?[\s.-]?\d{3}[\s.-]?\d{4}")

print("=== Flexible phone regex ===")
for text in texts:
    matches = re.findall(flexible_regex, text)
    status = "✓" if matches else "✗"
    print(f"  {status} {matches}  ← ...{text[-35:]}")

=== Flexible phone regex ===
  ✓ ['202-456-1111']  ← ...hite house, please use 202-456-1111
  ✓ ['(202)456-1111']  ← ...ite house, please use (202)456-1111
  ✓ ['(202) 456-1111']  ← ...te house, please use (202) 456-1111
  ✓ ['2024561111']  ← ... White house, please use 2024561111
  ✓ ['202.456.1111']  ← ...hite house, please use 202.456.1111
  ✓ ['202 456 1111']  ← ...hite house, please use 202 456 1111


In [55]:
# ============================================================
# PRACTICE EXERCISES: Extract structured entities with regex
# ============================================================

# --- Exercise 1: Product Codes (format: PRD-XXXX) ---
product_text = """1. Premium Laptop (PRD-2345) is on sale.
2. PRD-9876 is the latest smartphone.
3. Our bestseller, PRD-1200, has great reviews.
4. Customers love PRD-4507 and PRD-8765."""

# Pattern explanation:
# PRD-   → literal text "PRD-"
# \d{4}  → exactly 4 digits
product_pattern = r'PRD-\d{4}'
products = re.findall(product_pattern, product_text)
print(f"Product codes found: {products}")

# --- Exercise 2: Email Validation ---
emails = [
    "john.doe@email.com",           # valid
    "jane_doe123@business.net",      # valid
    "coolcoolcool@@gmail.com",       # invalid: double @
    "michael@company",               # invalid: no TLD
    "@nodomain.com",                 # invalid: no username
]

# Pattern explanation:
# ^               → start of string
# [a-zA-Z0-9._-]+ → username (letters, digits, dots, underscores, hyphens)
# @               → literal @ symbol
# [a-zA-Z0-9.-]+  → domain name
# \.[a-zA-Z]{2,}  → dot + TLD (at least 2 letters, e.g., .com, .net)
# $               → end of string
email_pattern = r'^[a-zA-Z0-9._-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'

print("\nEmail validation:")
for email in emails:
    valid = "✓ valid" if re.match(email_pattern, email) else "✗ invalid"
    print(f"  {valid}: {email}")

# --- Exercise 3: Dollar Amounts ---
finance_text = """Total revenue: $10,000.50 and net profit: $500.
USD 1,250.75 was spent on marketing.
Investment in new tech: USD 4.25.
Refund issued: $3000."""

# Pattern explanation:
# (?:\$|USD)       → either $ or USD (non-capturing group)
# \s?              → optional space after $ or USD
# \d{1,3}          → 1-3 starting digits
# (?:,\d{3})*      → optional comma-separated thousands
# (?:\.\d{2})?     → optional decimal with 2 digits
money_pattern = r'(?:\$|USD)\s?\d{1,3}(?:,\d{3})*(?:\.\d{2})?'
amounts = re.findall(money_pattern, finance_text)
print(f"\nDollar amounts found: {amounts}")

Product codes found: ['PRD-2345', 'PRD-9876', 'PRD-1200', 'PRD-4507', 'PRD-8765']

Email validation:
  ✓ valid: john.doe@email.com
  ✓ valid: jane_doe123@business.net
  ✗ invalid: coolcoolcool@@gmail.com
  ✗ invalid: michael@company
  ✗ invalid: @nodomain.com

Dollar amounts found: ['$10,000.50', '$500', 'USD 1,250.75', 'USD 4.25', '$300']


---
## Part 5: spaCy EntityRuler & Dictionary-Based NER

Sometimes you have a **known list of entities** (e.g., all NBA teams,
all ICD-10 medical codes, Fortune 500 companies). The EntityRuler lets
you define custom entity patterns and add them to spaCy's pipeline.

**Two approaches:**
1. **Exact phrase matching** — for dictionary/lookup table entities
2. **Pattern matching** — for regex-like patterns within spaCy

In [ ]:
# ============================================================
# APPROACH 1: Dictionary-Based NER with EntityRuler
# ============================================================
# Use case: You have a list of domain-specific terms to recognize

# Start with a blank model (no pretrained NER)
nlp_custom = spacy.blank("en")

# Create an EntityRuler — this is a rule-based component that
# matches patterns and assigns entity labels
ruler = nlp_custom.add_pipe("entity_ruler")

# Define a finance dictionary: term → label
# Each entry creates a pattern rule
finance_dict = {
    "dividend": "FINANCE_TERM",
    "dividends": "FINANCE_TERM",
    "earnings": "FINANCE_TERM",
    "revenue": "FINANCE_TERM",
    "EBITDA": "FINANCE_TERM",
    "gross margin": "FINANCE_TERM",
    "operating income": "FINANCE_TERM",
    "free cash flow": "FINANCE_TERM",
    "price-to-earnings ratio": "FINANCE_TERM",
    "S&P 500": "INDEX",
    "Dow Jones": "INDEX",
    "NASDAQ": "INDEX",
}

# Convert dictionary to spaCy pattern format
# Each pattern needs: {"label": ..., "pattern": ...}
# For exact phrases, use a simple string pattern
patterns = [{"label": label, "pattern": term}
            for term, label in finance_dict.items()]

ruler.add_patterns(patterns)
print(f"Added {len(patterns)} patterns to EntityRuler")

# Test it
test_texts = [
    "The company's revenue and EBITDA exceeded expectations.",
    "Investors focused on dividends and free cash flow.",
    "The S&P 500 and NASDAQ both hit record highs.",
]

for text in test_texts:
    doc = nlp_custom(text)
    print(f"\nText: {text}")
    for ent in doc.ents:
        print(f"  → {ent.text:<30} {ent.label_}")

Added 12 patterns to EntityRuler

Text: The company's revenue and EBITDA exceeded expectations.
  → revenue                        FINANCE_TERM
  → EBITDA                         FINANCE_TERM

Text: Investors focused on dividends and free cash flow.
  → dividends                      FINANCE_TERM
  → free cash flow                 FINANCE_TERM

Text: The S&P 500 and NASDAQ both hit record highs.
  → S&P 500                        INDEX
  → NASDAQ                         INDEX


In [57]:
# ============================================================
# APPROACH 2: Regex Patterns in EntityRuler
# ============================================================
# Combine the power of regex with spaCy's pipeline
# Use case: dates, IDs, codes that follow a pattern

nlp2 = spacy.load("en_core_web_sm")

# Add EntityRuler BEFORE the default NER so our rules take priority
ruler2 = nlp2.add_pipe("entity_ruler", before="ner")

# Define patterns using spaCy's token-level pattern matching
# Each dict in the "pattern" list matches one token
patterns2 = [
    # Match dates like 12/25/2020 (using regex on individual tokens)
    {"label": "DATE_CUSTOM", "pattern": [{"TEXT": {"REGEX": r"\d{1,2}/\d{1,2}/\d{4}"}}]},

    # Match dates like 2021-06-15
    {"label": "DATE_CUSTOM", "pattern": [{"TEXT": {"REGEX": r"\d{4}-\d{2}-\d{2}"}}]},

    # Match product codes like PRD-1234
    {"label": "PRODUCT_CODE", "pattern": [{"TEXT": {"REGEX": r"PRD-\d{4}"}}]},
]

ruler2.add_patterns(patterns2)

# Test with mixed text
test_text = "We ordered PRD-2345 on 12/25/2020 and it shipped by 2021-01-02."
doc = nlp2(test_text)

print(f"Text: {test_text}\n")
for ent in doc.ents:
    print(f"  → {ent.text:<20} {ent.label_}")

Text: We ordered PRD-2345 on 12/25/2020 and it shipped by 2021-01-02.

  → PRD-2345             PRODUCT_CODE
  → 12/25/2020           DATE_CUSTOM
  → 2021-01-02           DATE


---
## Part 6: NER Data Formats — CoNLL, JSONL, and Label Studio JSON

NER models need structured training data. Three key formats:

### CoNLL Format (IOB2 tags)
Each line has one token and its tag, separated by a space/tab.
Sentences are separated by blank lines.
```
Thomas    B-PERSON
Jefferson I-PERSON
writes    O
```

### JSONL Format (character spans)
Each line is a JSON object with the text and entity positions.
```json
{"text": "Thomas Jefferson writes...", "label": [[0, 16, "PERSON"]]}
```

### Label Studio JSON Format
Label Studio uses its own JSON structure with `data`, `annotations`, and `result` fields.
Each annotation result contains `start`, `end`, `text`, and `labels`.
```json
{
  "data": {"text": "Thomas Jefferson writes..."},
  "annotations": [{"result": [
    {"value": {"start": 0, "end": 16, "text": "Thomas Jefferson", "labels": ["PERSON"]},
     "from_name": "label", "to_name": "text", "type": "labels"}
  ]}]
}
```

**Label Studio** (our primary annotation tool) exports in JSON format.
**Doccano** (backup tool) exports in JSONL format.
Many ML libraries (like sklearn-crfsuite) prefer CoNLL format.
You'll need to convert between them.

In [58]:
# ============================================================
# EXPORT spaCy PREDICTIONS AS CoNLL FORMAT
# ============================================================
# This creates training data from spaCy's predictions
# Useful for bootstrapping annotation (pre-labeling)

import json

texts = [
    "Thomas Jefferson writes the Declaration of Independence",
    "My friend Tom works at Google",
    "Apple announced record revenue of $81.8 billion in Q3 2024",
]

def export_conll(texts, nlp, output_file):
    """
    Export NER predictions in CoNLL (IOB2) format.

    Each token gets a tag:
      B-LABEL → first token of an entity
      I-LABEL → continuation tokens of an entity
      O       → not an entity (Outside)
    """
    with open(output_file, "w", encoding="utf-8") as f:
        for text in texts:
            doc = nlp(text)
            for sent in doc.sents:
                for token in sent:
                    # Determine the IOB tag for this token
                    if token.ent_iob_ == "O":
                        tag = "O"
                    else:
                        # ent_iob_ is "B" or "I", ent_type_ is the label
                        tag = f"{token.ent_iob_}-{token.ent_type_}"
                    f.write(f"{token.text}\t{tag}\n")
                f.write("\n")  # blank line between sentences

    print(f"CoNLL file written to: {output_file}")

export_conll(texts, nlp, "output.conll")

# Display the file contents
print("\n--- CoNLL File Contents ---")
with open("output.conll") as f:
    print(f.read())

CoNLL file written to: output.conll

--- CoNLL File Contents ---
Thomas	B-PERSON
Jefferson	I-PERSON
writes	O
the	B-WORK_OF_ART
Declaration	I-WORK_OF_ART
of	I-WORK_OF_ART
Independence	I-WORK_OF_ART

My	O
friend	O
Tom	B-PERSON
works	O
at	O
Google	B-ORG

Apple	B-ORG
announced	O
record	O
revenue	O
of	O
$	B-MONEY
81.8	I-MONEY
billion	I-MONEY
in	O
Q3	B-DATE
2024	I-DATE




In [59]:
# ============================================================
# EXPORT spaCy PREDICTIONS AS LABEL STUDIO JSON FORMAT
# ============================================================
# Label Studio can import pre-annotations in its JSON format.
# This lets you upload model predictions for human review.
#
# Label Studio JSON structure:
#   - "data": contains the raw text (key must match your labeling config)
#   - "predictions": list of pre-annotations
#   - Each prediction has "result": list of entity spans
#   - Each span has "value" with start, end, text, labels

def export_label_studio_json(texts, nlp, output_file):
    """
    Export NER predictions in Label Studio JSON format for import.

    This creates pre-annotated tasks that can be imported into
    Label Studio for human review and correction.

    Args:
        texts: list of raw text strings
        nlp: spaCy NLP model
        output_file: path to write JSON file
    """
    tasks = []
    for text in texts:
        doc = nlp(text)

        # Build the annotation results list
        results = []
        for ent in doc.ents:
            results.append({
                "from_name": "label",       # must match your Label Studio config
                "to_name": "text",           # must match your Label Studio config
                "type": "labels",
                "value": {
                    "start": ent.start_char,
                    "end": ent.end_char,
                    "text": ent.text,
                    "labels": [ent.label_],
                },
            })

        task = {
            "data": {"text": text},
            "predictions": [{"result": results}] if results else [],
        }
        tasks.append(task)

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(tasks, f, indent=2)

    print(f"Label Studio JSON written to: {output_file} ({len(tasks)} tasks)")

export_label_studio_json(texts, nlp, "output_label_studio.json")

# Display the file contents
print("\n--- Label Studio JSON (first task) ---")
with open("output_label_studio.json") as f:
    data = json.load(f)
    print(json.dumps(data[0], indent=2))

Label Studio JSON written to: output_label_studio.json (3 tasks)

--- Label Studio JSON (first task) ---
{
  "data": {
    "text": "Thomas Jefferson writes the Declaration of Independence"
  },
  "predictions": [
    {
      "result": [
        {
          "from_name": "label",
          "to_name": "text",
          "type": "labels",
          "value": {
            "start": 0,
            "end": 16,
            "text": "Thomas Jefferson",
            "labels": [
              "PERSON"
            ]
          }
        },
        {
          "from_name": "label",
          "to_name": "text",
          "type": "labels",
          "value": {
            "start": 24,
            "end": 55,
            "text": "the Declaration of Independence",
            "labels": [
              "WORK_OF_ART"
            ]
          }
        }
      ]
    }
  ]
}


In [60]:
# ============================================================
# CONVERT BETWEEN FORMATS: Label Studio JSON ↔ CoNLL / JSONL
# ============================================================
# After annotating in Label Studio, you export as JSON.
# For model training, you often need CoNLL (IOB2) format.

def label_studio_to_iob2(ls_data):
    """
    Convert Label Studio JSON export to IOB2 token-label pairs.

    Handles the character-to-token alignment:
    - Tokenize by whitespace (preserving character positions)
    - Map character-level entity spans to token-level IOB tags

    Args:
        ls_data: list of Label Studio task dicts (parsed JSON)

    Returns:
        List of sentences, each a list of (token, tag) tuples
    """
    import re as regex_module
    all_sentences = []

    for task in ls_data:
        text = task["data"]["text"]

        # Get annotations (could be in "annotations" or "predictions")
        annotations = task.get("annotations", task.get("predictions", []))
        if not annotations:
            continue

        # Collect all entity spans from the first annotation
        results = annotations[0].get("result", [])
        entities = []
        for r in results:
            if r.get("type") == "labels":
                val = r["value"]
                entities.append((val["start"], val["end"], val["labels"][0]))

        entities.sort(key=lambda x: x[0])

        # Tokenize and track character positions
        tokens = []
        token_spans = []
        for match in regex_module.finditer(r'\S+', text):
            tokens.append(match.group())
            token_spans.append((match.start(), match.end()))

        # Assign IOB tags
        tags = ["O"] * len(tokens)
        for ent_start, ent_end, label in entities:
            entity_started = False
            for i, (tok_start, tok_end) in enumerate(token_spans):
                if tok_start >= ent_start and tok_start < ent_end:
                    tags[i] = f"B-{label}" if not entity_started else f"I-{label}"
                    entity_started = True

        all_sentences.append(list(zip(tokens, tags)))

    return all_sentences

# Demo: convert our exported Label Studio JSON back to IOB2
with open("output_label_studio.json") as f:
    ls_data = json.load(f)

sentences_iob = label_studio_to_iob2(ls_data)

print("Label Studio JSON → IOB2 conversion:")
for i, sent in enumerate(sentences_iob[:2]):
    print(f"\nSentence {i+1}:")
    for token, tag in sent:
        marker = " ←" if tag != "O" else ""
        print(f"  {token:<25} {tag}{marker}")

# Also export as CoNLL file for CRF training
def iob2_to_conll(sentences, output_file):
    """Write IOB2 sentences to CoNLL format file."""
    with open(output_file, "w") as f:
        for sent in sentences:
            for token, tag in sent:
                f.write(f"{token}\t{tag}\n")
            f.write("\n")
    print(f"\nCoNLL file written to: {output_file}")

iob2_to_conll(sentences_iob, "from_label_studio.conll")

Label Studio JSON → IOB2 conversion:

Sentence 1:
  Thomas                    B-PERSON ←
  Jefferson                 I-PERSON ←
  writes                    O
  the                       B-WORK_OF_ART ←
  Declaration               I-WORK_OF_ART ←
  of                        I-WORK_OF_ART ←
  Independence              I-WORK_OF_ART ←

Sentence 2:
  My                        O
  friend                    O
  Tom                       B-PERSON ←
  works                     O
  at                        O
  Google                    B-ORG ←

CoNLL file written to: from_label_studio.conll


In [61]:
# ============================================================
# LOAD LABEL STUDIO EXPORT AND VISUALIZE
# ============================================================
# After annotating in Label Studio, export as JSON.
# Here's how to load it and visualize the annotations.

def load_and_visualize_label_studio(filepath):
    """
    Load a Label Studio JSON export and display entities using displaCy.

    Args:
        filepath: path to Label Studio JSON export file
    """
    with open(filepath) as f:
        tasks = json.load(f)

    for task in tasks:
        text = task["data"]["text"]

        # Get annotations (from completed annotations or predictions)
        annotations = task.get("annotations", task.get("predictions", []))
        if not annotations:
            continue

        results = annotations[0].get("result", [])

        # Convert to displaCy format
        ents = []
        for r in results:
            if r.get("type") == "labels":
                val = r["value"]
                ents.append({
                    "start": val["start"],
                    "end": val["end"],
                    "label": val["labels"][0],
                })

        doc_data = {"text": text, "ents": ents, "title": None}
        displacy.render(doc_data, style="ent", manual=True, jupyter=True)

# Visualize our exported annotations
print("Visualizing Label Studio annotations:\n")
load_and_visualize_label_studio("output_label_studio.json")

Visualizing Label Studio annotations:



---
## Summary: Notebook 1

| Tool | Best For | Limitations |
|------|----------|-------------|
| **spaCy** | Production NER, fast inference | Fixed entity types in pretrained model |
| **NLTK** | Understanding the NLP pipeline | Low accuracy, few entity types |
| **HuggingFace** | Best accuracy with Transformers | Slower, needs GPU for large models |
| **Regex** | Structured patterns (phone, email, codes) | Can't handle ambiguous entities |
| **EntityRuler** | Known dictionaries + spaCy pipeline | Manual maintenance of entity lists |

### Next Steps
- **Notebook 2**: Train your own NER models (CRF, Transformer)
- **Notebook 3**: Use LLMs for zero-shot NER and build an end-to-end pipeline with Label Studio